# Mini Czech Benchmark

This notebook evaluates models by running a text generation pipeline over four datasets: `agree`, `czech_news`, `klokanek` and `ctkfacts`, either *full* data or *mini* (random 200 rows) or *tiny* ([selected](https://arxiv.org/abs/2402.14992) 100 rows).

It can be run via [Papermill](https://github.com/nteract/papermill) as follows:

```
  model="meta-llama/Llama-3.2-1B-Instruct"  # or other HuggingFace model
  hf_token="** Your HuggingFace read access token **"
  papermill minicz_bench.ipynb output.ipynb -p MODEL "$model" -p HF_TOKEN "$hf_token" -p DATA_SIZE "mini" -p OUTPUT_DIR "" -k python3
```

[MiniCzechBenchmark](https://github.com/simecek/MiniCzechBenchmark) is a small subset selected from [CzechBench](https://gitlab.com/jirkoada/czech-bench) benchmark suited for fast model assessment.

In [43]:
# papermill parameters

MODEL = 'ibm-granite/granite-4.0-micro' # hf hub model, e.g. mistralai/Mistral-7B-Instruct-v0.3

MESSAGES  = 'simplemessages' # Choose one of: 'simplemessages' or 'justprompt' or 'useronly'

DATA_SIZE = 'mini' # Choose one of: 'full' or 'mini' or 'tiny'

OUTPUT_DIR = 'lesson08\miniczechbenchmark\tested'  # folder to export metrics and outputs (None or '' to avoid saving)

PIPELINE_TYPE = 'noSampling'  # type of text-generation pipeline; currently only 'noSampling' supported

HF_TOKEN = ''  # HF token needed to access gated models

## Text-generation Pipeline

In [2]:
# from huggingface_hub import login
# login(HF_TOKEN)

In [3]:
import bitsandbytes
import torch
from transformers import AutoTokenizer, pipeline
from datasets import load_dataset
import pandas as pd
import time

MAX_NEW_TOKENS = {
    'agree': 2,
    'czech_news': 2,
    'klokanek': 2,
    'ctkfacts': 2
}

def message_function(message_strategy, user_prompts, system_prompts):
    if message_strategy == 'simplemessages':
        messages = [[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ] for system_prompt, user_prompt  in zip(system_prompts, user_prompts)]
    elif message_strategy == 'justprompt':
        messages = [f"{system_prompt}\n\n{user_prompt}" for system_prompt, user_prompt in zip(system_prompts, user_prompts)]
    elif message_strategy == 'useronly':
        messages = [[
            {"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"},
        ] for system_prompt, user_prompt  in zip(system_prompts, user_prompts)]
    else:
        raise('Message strategy not implemeted')
        
    return messages

def cleaning_function(raw_outputs):
    return [x[0]['generated_text'][:3].strip().replace(")", "").replace(".", "") for x in raw_outputs]

if DATA_SIZE == "full":
    DATASETS = {
        'agree': 'simecek/agree',
        'czech_news': 'simecek/czech_news',
        'klokanek': 'simecek/klokanek',
        'ctkfacts': 'simecek/ctkfacts'}
elif  DATA_SIZE == "mini":
    DATASETS = {
        'agree': 'simecek/mini_agree',
        'czech_news': 'simecek/mini_czech_news',
        'klokanek': 'simecek/mini_klokanek',
        'ctkfacts': 'simecek/mini_ctkfacts'}
elif  DATA_SIZE == "tiny":
    raise(f"Data size {DATA_SIZE} not implemeted")
else:
    raise(f"Data size {DATA_SIZE} not implemeted")

class Timer:
    def __init__(self, name="Elapsed time", storage_dict=None, key=None):
        self.name = name
        self.storage_dict = storage_dict
        self.key = key
    
    def __enter__(self):
        self.start_time = time.time()
        return self  
    
    def __exit__(self, *args):
        self.end_time = time.time()
        self.elapsed = self.end_time - self.start_time
        if self.storage_dict is not None and self.key is not None:
            self.storage_dict[self.key] = self.elapsed

2025-11-09 17:50:40.968466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-09 17:50:40.984399: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-09 17:50:40.989327: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-09 17:50:41.006534: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-09 17:50:41.943908: W tensorflow/compiler/tf2

In [4]:
# define the pipeline

tok = AutoTokenizer.from_pretrained(MODEL)  # needed by granite models

pipe = pipeline(
    "text-generation", 
    model=MODEL,
    tokenizer=tok,
    model_kwargs={"torch_dtype": torch.bfloat16}, 
    device_map="auto",
    do_sample=False,
    temperature=0,
    pad_token_id=tok.eos_token_id,
    trust_remote_code=True  # for Phi-3.5-Moe
)

# Explicitly set pad_token_id to eos_token_id to prevent the warning
pipe.model.config.pad_token_id = pipe.model.config.eos_token_id

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## Evaluation

In [5]:
raw_outputs = {}  # raw outputs from llm
clean_outputs = {}  # after cleaning
dfs = {}  # dataframe comparing clean_outputs to correct answers
metrics = {}  # overall summaries for each dataset
timing_results = {} # elapsed time

### AGREE

In [6]:
dataset_name = 'agree'

dt = load_dataset(DATASETS[dataset_name])

In [7]:
messages = message_function(MESSAGES, dt['train']['user_prompt'], dt['train']['system_prompt'])

In [8]:
tmp = pipe(messages[:5], return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])
tmp

[[{'generated_text': '4'}],
 [{'generated_text': '1'}],
 [{'generated_text': '1'}],
 [{'generated_text': '4'}],
 [{'generated_text': '4'}]]

In [9]:
with Timer(f"Processing {dataset_name}", storage_dict=timing_results, key=dataset_name):
    raw_outputs[dataset_name] = pipe(messages, return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])

In [10]:
clean_outputs[dataset_name] = cleaning_function(raw_outputs[dataset_name])

In [11]:
dfs[dataset_name] = pd.DataFrame({
    'correct_answer_minus_1': dt['train']['answer_idx'],
    'answer': clean_outputs[dataset_name],
})

dfs[dataset_name]

,correct_answer_minus_1,answer
0,0,4
1,1,1
2,2,1
3,4,4
4,2,4
...,...,...
195,3,1
196,2,3
197,4,3
198,4,3


In [12]:
dfs[dataset_name]['valid'] = dfs[dataset_name].answer.isin(['1', '2', '3', '4', '5'])
dfs[dataset_name]['correct'] = [(x in ['1', '2', '3', '4', '5']) and int(x) == int(y)+1 for x,y in zip(dfs[dataset_name].answer, dfs[dataset_name].correct_answer_minus_1)]

In [13]:
# correct answers vs valid answers
metrics[dataset_name] = (dfs[dataset_name]['valid'].mean().item(), dfs[dataset_name].correct.mean().item())
metrics[dataset_name]

(1.0, 0.335)

### CZECH NEWS

In [14]:
dataset_name = 'czech_news'

dt = load_dataset(DATASETS[dataset_name])

In [15]:
messages = message_function(MESSAGES, dt['train']['user_prompt'], dt['train']['system_prompt'])

In [16]:
tmp = pipe(messages[:5], return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])
tmp

[[{'generated_text': '1'}],
 [{'generated_text': '1'}],
 [{'generated_text': '3'}],
 [{'generated_text': '5'}],
 [{'generated_text': '5'}]]

In [17]:
with Timer(f"Processing {dataset_name}", storage_dict=timing_results, key=dataset_name):
    raw_outputs[dataset_name] = pipe(messages, return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])

In [18]:
clean_outputs[dataset_name] = cleaning_function(raw_outputs[dataset_name])

In [19]:
dfs[dataset_name] = pd.DataFrame({
    'correct_answer': dt['train']['category'],
    'answer': clean_outputs[dataset_name],
})

dfs[dataset_name]

,correct_answer,answer
0,4,1
1,4,1
2,3,3
3,5,5
4,5,5
...,...,...
195,5,5
196,5,2
197,1,1
198,5,5


In [20]:
dfs[dataset_name]['valid'] = dfs[dataset_name].answer.isin(['1', '2', '3', '4', '5'])
dfs[dataset_name]['correct'] = dfs[dataset_name].answer.apply(str) == dfs[dataset_name].correct_answer.apply(str)

In [21]:
# correct answers vs valid answers
metrics[dataset_name] = (dfs[dataset_name]['valid'].mean().item(), dfs[dataset_name].correct.mean().item())
metrics[dataset_name]

(1.0, 0.59)

### KLOKANEK

In [22]:
dataset_name = 'klokanek'

dt = load_dataset(DATASETS[dataset_name])

In [23]:
messages = message_function(MESSAGES, dt['train']['user_prompt'], dt['train']['system_prompt'])

In [24]:
tmp = pipe(messages[:5], return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])
tmp

[[{'generated_text': 'A'}],
 [{'generated_text': 'C'}],
 [{'generated_text': 'C'}],
 [{'generated_text': 'C'}],
 [{'generated_text': 'C'}]]

In [25]:
with Timer(f"Processing {dataset_name}", storage_dict=timing_results, key=dataset_name):
    raw_outputs[dataset_name] = pipe(messages, return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])

In [26]:
clean_outputs[dataset_name] = cleaning_function(raw_outputs[dataset_name])

In [27]:
dfs[dataset_name] = pd.DataFrame({
    'correct_answer': dt['train']['correct_answer'],
    'answer': clean_outputs[dataset_name],
})

dfs[dataset_name]

,correct_answer,answer
0,B,A
1,B,C
2,D,C
3,A,C
4,E,C
...,...,...
195,E,C
196,C,C
197,D,B
198,D,C


In [28]:
dfs[dataset_name]['valid'] = dfs[dataset_name].answer.str.lower().isin(['a', 'b', 'c', 'd', 'e'])
dfs[dataset_name]['correct'] = dfs[dataset_name].answer.str.lower() == dfs[dataset_name].correct_answer.str.lower()

In [29]:
# correct answers vs valid answers
metrics[dataset_name] = (dfs[dataset_name]['valid'].mean().item(), dfs[dataset_name].correct.mean().item())
metrics[dataset_name]

(1.0, 0.245)

### CTK Facts

In [30]:
dataset_name = 'ctkfacts'

dt = load_dataset(DATASETS[dataset_name])

In [31]:
messages = message_function(MESSAGES, dt['train']['user_prompt'], dt['train']['system_prompt'])

In [32]:
tmp = pipe(messages[:5], return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])
tmp

[[{'generated_text': '1'}],
 [{'generated_text': '1'}],
 [{'generated_text': '1'}],
 [{'generated_text': '2'}],
 [{'generated_text': '2'}]]

In [33]:
with Timer(f"Processing {dataset_name}", storage_dict=timing_results, key=dataset_name):
    raw_outputs[dataset_name] = pipe(messages, return_full_text=False, max_new_tokens=MAX_NEW_TOKENS[dataset_name])

In [34]:
clean_outputs[dataset_name] = cleaning_function(raw_outputs[dataset_name])

In [35]:
dfs[dataset_name] = pd.DataFrame({
    'correct_answer': dt['train']['label'],
    'answer': clean_outputs[dataset_name],
})

dfs[dataset_name]

,correct_answer,answer
0,1,1
1,2,1
2,2,1
3,2,2
4,1,2
...,...,...
195,2,2
196,1,1
197,0,0
198,2,1


In [36]:
dfs[dataset_name]['valid'] = dfs[dataset_name].answer.isin(['0', '1', '2'])
dfs[dataset_name]['correct'] = dfs[dataset_name].answer == dfs[dataset_name].correct_answer.apply(str)

In [37]:
# correct answers vs valid answers
metrics[dataset_name] = (dfs[dataset_name]['valid'].mean().item(), dfs[dataset_name].correct.mean().item())
metrics[dataset_name]

(1.0, 0.645)

## Metrics & Export

In [38]:
metrics

{'agree': (1.0, 0.335),
 'czech_news': (1.0, 0.59),
 'klokanek': (1.0, 0.245),
 'ctkfacts': (1.0, 0.645)}

In [39]:
timing_results

{'agree': 20.468610763549805,
 'czech_news': 35.49202513694763,
 'klokanek': 27.003090381622314,
 'ctkfacts': 43.90155005455017}

In [40]:
RUN_NAME = f"{MODEL.split('/')[1]}_{DATA_SIZE}_{MESSAGES}_{PIPELINE_TYPE}"
RUN_NAME

'granite-4.0-micro_mini_simplemessages_noSampling'

In [44]:
import pickle
import gzip

objects_to_save = {
    'model': MODEL,
    'datasets': DATASETS,
    'raw_outputs': raw_outputs,
    'dfs': dfs,
    'metrics': metrics,
    'timing_results': timing_results,
}

if OUTPUT_DIR:
    with gzip.open(f"{OUTPUT_DIR}/{RUN_NAME}.pkl.gz", "wb") as file:
        pickle.dump(objects_to_save, file)

In [ ]:
#with gzip.open(f"{OUTPUT_DIR}/{RUN_NAME}.pkl.gz", "rb") as file:
#    loaded_objects = pickle.load(file)